# Auditoría de la ingesta de Persona1

Equipo 39. Esta copia se ejecuta localmente por autorización de Persona1. Revisa los doce meses de MiBici y los archivos generados por `dataset.py`. No entrena modelos ni sustituye la reproducción oficial del equipo.

Antes de comenzar deben existir `references/data_manifest.json`, los CSV locales y `data/processed/stations.parquet`. Si faltan, termina primero la ingesta. Los datos permanecen fuera de GitHub.

## 1 Comprobar el entorno

Primero se consulta la versión disponible. Este notebook no instala programas. Si falta pandas o pyarrow, prepara el entorno del proyecto usando su `requirements.txt` y vuelve a abrirlo con ese Python.

In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "references/data_manifest.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "references/data_manifest.json").exists(), "Primero completa la ingesta"
sys.path.insert(0, str(ROOT))
from mibici_equipo39 import config as C
print("Python:", sys.version.split()[0])
print("pandas:", importlib.metadata.version("pandas"))
print("pyarrow:", importlib.metadata.version("pyarrow"))
print("Responsable: Persona1 | Equipo 39 | ejecución local")

Python: 3.11.9
pandas: 2.2.3
pyarrow: 19.0.1
Responsable: Persona1 | Equipo 39 | ejecución local


## 2 Revisar el periodo y las decisiones de configuración

Se conservan doce meses completos. La partición que implementará Persona2 es aleatoria por días completos y estratificada por mes. La semilla se toma de `config.py`. Una evaluación aleatoria no demuestra el desempeño en meses futuros.

In [2]:
settings = {"inicio": C.SOURCE_START, "fin_exclusivo": C.SOURCE_END_EXCLUSIVE,
            "meses": len(C.SOURCE_URLS), "seed": C.SEED,
            "train": C.TRAIN_SIZE, "validation": C.VAL_SIZE, "test": C.TEST_SIZE,
            "particion": C.SPLIT_STRATEGY}
assert settings["meses"] == 12 and C.SEED == 39
print(json.dumps(settings, indent=2))

{
  "inicio": "2025-10-01",
  "fin_exclusivo": "2026-10-01",
  "meses": 12,
  "seed": 39,
  "train": 0.7,
  "validation": 0.15,
  "test": 0.15,
  "particion": "random_days_month_stratified"
}


## 3 Verificar las trece fuentes

Se calcula SHA256 de cada archivo descargado y se compara con el manifiesto congelado. Una diferencia detiene la auditoría. La primera adquisición en esta máquina se realizó desde los enlaces del portal porque requests recibió HTTP 406. El hash permite comprobar si otra persona obtuvo la misma versión.

In [3]:
manifest = json.loads((ROOT / "references/data_manifest.json").read_text(encoding="utf-8"))
checks = []
for source in manifest["sources"]:
    folder = "external" if source["name"].startswith("nomenclatura") else "raw"
    path = ROOT / "data" / folder / source["name"]
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    ok = h.hexdigest() == source["sha256"]
    assert ok, "Fuente distinta: " + source["name"]
    checks.append({"archivo": source["name"], "bytes": path.stat().st_size, "hash_coincide": ok})
assert len(checks) == 13
print(pd.DataFrame(checks).to_string(index=False))
print("data_version:", manifest["data_version"])

                   archivo    bytes  hash_coincide
  nomenclatura_2026_09.csv    40841           True
datos_abiertos_2025_10.csv 36442309           True
datos_abiertos_2025_11.csv 32997087           True
datos_abiertos_2025_12.csv 29089463           True
datos_abiertos_2026_01.csv 30985189           True
datos_abiertos_2026_02.csv 28953901           True
datos_abiertos_2026_03.csv 32640591           True
datos_abiertos_2026_04.csv 30523683           True
datos_abiertos_2026_05.csv 27427394           True
datos_abiertos_2026_06.csv 30826027           True
datos_abiertos_2026_07.csv 33696291           True
datos_abiertos_2026_08.csv 34158210           True
datos_abiertos_2026_09.csv 42566896           True
data_version: e9de13df5e9bdc1fde98c042f700e1a7c568ad85df5a303a96a78b3d4d45dbb7


## 4 Revisar los conteos reales por mes

Estas cifras provienen de los archivos oficiales procesados. Los conteos sintéticos de pytest no se usan en esta sección.

In [4]:
quality = json.loads((ROOT / "reports/calidad_datos.json").read_text(encoding="utf-8"))
monthly = pd.read_csv(ROOT / "reports/calidad_datos_por_mes.csv")
assert monthly["raw_rows"].sum() == quality["raw_rows"]
assert monthly["retained_rows"].sum() == quality["retained_rows"]
print(json.dumps(quality, indent=2))
print(monthly[["month", "raw_rows", "retained_rows", "duplicates"]].to_string(index=False))

{
  "months": 12,
  "data_version": "e9de13df5e9bdc1fde98c042f700e1a7c568ad85df5a303a96a78b3d4d45dbb7",
  "raw_rows": 4497877,
  "retained_rows": 4497119,
  "catalog_rows": 484,
  "valid_capacity_stations": 480,
  "origin_stations": 471,
  "destination_stations": 471,
  "union_stations": 471,
  "source_bytes": 390347882
}
  month  raw_rows  retained_rows  duplicates
2025-10    420064         420064           0
2025-11    380328         380328           0
2025-12    335420         335420           0
2026-01    357180         357180           0
2026-02    333756         333756           0
2026-03    376243         376243           0
2026-04    351851         351851           0
2026-05    315895         315895           0
2026-06    355303         355303           0
2026-07    388257         388257           0
2026-08    393524         393524           0
2026-09    490056         489298         758


## 5 Conciliar las exclusiones

Cada fila descartada recibe un motivo excluyente. Los eventos de salida y llegada se informan por separado: una salida válida no requiere conocer el final futuro del viaje.

In [5]:
reasons = ["missing_trip_id", "duplicates", "invalid_start", "outside_range",
           "missing_station", "invalid_end", "end_before_start"]
assert (monthly["raw_rows"] == monthly["retained_rows"] + monthly[reasons].sum(axis=1)).all()
print(monthly[reasons].sum().to_string())
print("Conciliación correcta en los doce meses")

missing_trip_id       0
duplicates          758
invalid_start         0
outside_range         0
missing_station       0
invalid_end           0
end_before_start      0
Conciliación correcta en los doce meses


## 6 Revisar la capacidad para P2

Se usa `dpcapacity` del catálogo publicado, renombrada como `capacity`. Se excluyen de P2 las capacidades ausentes, cero, negativas o fraccionarias. El catálogo es una fotografía de septiembre de 2026: no prueba la capacidad histórica de cada estación.

Persona3 debe revisar la aproximación de ocupación inicial del 50% y los umbrales de 10% y 90%. Sin inventario observado ni movimientos de rebalanceo no se puede afirmar que una estación estuvo realmente vacía o llena.

In [6]:
stations = pd.read_parquet(ROOT / "data/processed/stations.parquet")
valid = stations["valid_capacity"]
assert valid.sum() == quality["valid_capacity_stations"]
summary = {"estaciones_catalogo": len(stations), "capacidad_valida": int(valid.sum()),
           "excluidas_P2": int((~valid).sum()),
           "capacidad_minima_valida": float(stations.loc[valid, "capacity"].min()),
           "capacidad_maxima_valida": float(stations.loc[valid, "capacity"].max())}
print(json.dumps(summary, indent=2))
print(stations.loc[valid, "capacity"].describe().to_string())

{
  "estaciones_catalogo": 484,
  "capacidad_valida": 480,
  "excluidas_P2": 4,
  "capacidad_minima_valida": 11.0,
  "capacidad_maxima_valida": 39.0
}
count        480.0
mean     17.022917
std       3.843697
min           11.0
25%           15.0
50%           15.0
75%           19.0
max           39.0


## 7 Guardar un resumen agregado

El reporte incluye hashes y conteos; no contiene viajes ni identificadores de usuario. Puede incorporarse al PR de ingesta después de revisar su contenido.

In [7]:
audit = {"executor": "persona1", "execution_environment": "local",
         "data_version": manifest["data_version"], "sources_verified": len(checks),
         "raw_rows": quality["raw_rows"], "retained_rows": quality["retained_rows"],
         "station_capacity": summary}
output = ROOT / "reports/auditoria_persona1.json"
output.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print("Reporte guardado en reports/auditoria_persona1.json")

Reporte guardado en reports/auditoria_persona1.json


## 8 Dependencias antes de continuar

Persona1 puede preparar y probar su ingesta sin esperar otro código. Para incorporarla necesita publicar primero el PR del scaffold y recibir la aprobación formal de Persona2. Después Persona2 debe reconstruir los datos y comparar `data_version` en el PR de ingesta.

La ejecución oficial espera las integraciones de P1 y variables de Persona2, P2 de Persona3 y P3 y Tracking de Persona4. Persona1 debe revisar formalmente el PR de Persona4. Las ejecuciones y revisiones de los otros integrantes deben realizarse desde sus propias cuentas.